# HelloML — Handwritten Digit OCR (DIDA)

**End-to-end multi-model comparison for optical character recognition**

| Item | Detail |
|------|--------|
| **Dataset** | DIDA (digits 0–9, 1,000 images each → 10,000 total) |
| **Task** | Multi-class classification (10 classes) |
| **Split** | 80% train / 20% test, stratified |
| **Models** | GaussianNB · Linear Regression (One-vs-All) · Logistic Regression · MLP |
| **Metrics** | Accuracy, Precision, Recall, F1 (macro), Confusion Matrix |
| **CV** | 5-fold stratified + `GridSearchCV` |

This notebook unifies the assignment requirements with an **enhanced preprocessing pipeline** (median denoise → K-Means binarization → bounding-box crop → scale & center on 28×28).


## 0. Environment setup

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


## 1. Imports

In [ ]:
from pathlib import Path
import time
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
os.environ["PYTHONWARNINGS"] = "ignore::sklearn.exceptions.ConvergenceWarning"

np.set_printoptions(precision=2)
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_colwidth", None)

print("Imports OK")


## 2. Load the DIDA dataset

Reads grayscale images from folders `0` … `9`.  
`files_per_folder` controls sample size (use a smaller value for faster experiments).


In [ ]:
def load_dataset(root_path, files_per_folder=1000, seed=42):
    """
    Load DIDA images from folders '0'..'9'.
    Returns X (list of arrays), y (labels).
    """
    root_path = Path(root_path)
    print(f"Loading data from: {root_path}")
    t0 = time.time()
    X, y = [], []
    rng = np.random.default_rng(seed)
    for digit in range(10):
        folder = root_path / str(digit)
        if not folder.exists():
            raise FileNotFoundError(f"Folder not found: {folder}")
        all_files = [p for p in folder.iterdir() if p.is_file()]
        if len(all_files) < files_per_folder:
            raise ValueError(f"Folder '{digit}' has {len(all_files)} images, need >= {files_per_folder}.")
        chosen = sorted(rng.choice(all_files, size=files_per_folder, replace=False))
        for path in chosen:
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"Warning: could not read {path}")
                continue
            X.append(img)
            y.append(digit)
    y = np.array(y, dtype=np.int64)
    print(f"Loaded {len(X)} images in {time.time() - t0:.1f}s")
    return X, y


## 3. Enhanced preprocessing

1. Median blur (3×3) — denoise
2. K-Means (k=2) — adaptive ink vs background
3. Bounding-box crop
4. Scale longest side → 20 px
5. Center on 28×28 canvas
6. Normalize to [0,1] + flatten → 784 features


In [ ]:
def binarize_center_resize(imgs, target_size=(28, 28)):
    H, W = target_size
    out = np.zeros((len(imgs), H, W), dtype=np.uint8)
    for i, img in enumerate(imgs):
        denoised = cv2.medianBlur(img, 3)
        pixels = denoised.reshape(-1, 1).astype(np.float64)
        kmeans = KMeans(n_clusters=2, n_init=10, max_iter=300, random_state=42)
        labels = kmeans.fit_predict(pixels)
        ink_label = int(np.argmin(kmeans.cluster_centers_))
        binary = (labels == ink_label).reshape(denoised.shape).astype(np.uint8) * 255
        coords = np.column_stack(np.where(binary > 0))
        if coords.shape[0] > 0:
            x, y, w, h = cv2.boundingRect(binary)
            crop = binary[y:y+h, x:x+w]
            scale = 20.0 / max(w, h)
            new_w = max(1, int(w * scale))
            new_h = max(1, int(h * scale))
            resized = cv2.resize(crop, (new_w, new_h), interpolation=cv2.INTER_AREA)
            canvas = np.zeros((H, W), dtype=np.uint8)
            sy = (H - new_h) // 2
            sx = (W - new_w) // 2
            canvas[sy:sy+new_h, sx:sx+new_w] = resized
            out[i] = canvas
        else:
            out[i] = cv2.resize(binary, (W, H))
    print(f"Preprocessed images: {out.shape}")
    return out

def normalize_flatten_split(imgs, y, test_size=0.20, seed=42):
    X = (imgs.astype(np.float32) / 255.0).reshape(len(imgs), -1)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )
    print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
    return X_train, X_test, y_train, y_test


## 4. Run load + preprocess

In [ ]:
root = Path.cwd() / "DIDA"
if not root.exists():
    raise FileNotFoundError(f"'DIDA' not found in {Path.cwd()}. Place the dataset folder here.")
X_raw, y = load_dataset(root, files_per_folder=1000)
X_proc = binarize_center_resize(X_raw)
X_train, X_test, y_train, y_test = normalize_flatten_split(X_proc, y)


## 5. Preview: raw vs preprocessed

In [ ]:
def show_before_after(raw_list, proc_arr, y, n_per_class=1, seed=0):
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(10, 2, figsize=(4, 12))
    for digit in range(10):
        idxs = np.where(y == digit)[0]
        idx = rng.choice(idxs)
        axes[digit, 0].imshow(raw_list[idx], cmap="gray")
        axes[digit, 0].set_title(f"raw {digit}", fontsize=8)
        axes[digit, 0].axis("off")
        axes[digit, 1].imshow(proc_arr[idx], cmap="gray")
        axes[digit, 1].set_title(f"proc {digit}", fontsize=8)
        axes[digit, 1].axis("off")
    plt.suptitle("Raw vs Preprocessed", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

show_before_after(X_raw, X_proc, y)


## 6. Model configurations

| Model | Notes |
|-------|--------|
| **GaussianNB** | Probabilistic baseline |
| **LinearReg_OvA** | OLS + One-vs-All |
| **LogisticReg** | Tune regularization C |
| **MLP** | Tune hidden layer sizes (min 2 layers) |


In [ ]:
def get_experiment_setup():
    return {
        "NaiveBayes": {"model": GaussianNB(), "params": {}},
        "LinearReg_OvA": {"model": OneVsRestClassifier(LinearRegression()), "params": {}},
        "LogisticReg": {
            "model": LogisticRegression(solver="lbfgs", max_iter=2000, random_state=42),
            "params": {"C": [1.0, 10.0]},
        },
        "MLP": {
            "model": MLPClassifier(random_state=42, max_iter=2000),
            "params": {
                "hidden_layer_sizes": [
                    (700, 200),
                    (500, 100, 10),
                    (300, 150, 100, 50, 10),
                ],
            },
        },
    }

scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
}
setup = get_experiment_setup()
print("Models:", list(setup.keys()))


## 7. Phase 1 — Hyperparameter tuning & 5-fold CV

`GridSearchCV` with cv=5, multi-metric scoring, refit on macro-F1.


In [ ]:
best_estimators = {}
cv_rows = []
cv_best_rows = []

print("=" * 60)
print("PHASE 1: GridSearchCV + Cross-Validation")
print("=" * 60)

for name, cfg in setup.items():
    print(f"\n>>> {name}")
    grid = GridSearchCV(
        estimator=cfg["model"],
        param_grid=cfg["params"],
        cv=5,
        scoring=scoring_metrics,
        refit="f1",
        n_jobs=-1,
        return_train_score=False,
        verbose=1,
    )
    t0 = time.time()
    grid.fit(X_train, y_train)
    print(f"    done in {time.time()-t0:.1f}s  |  best: {grid.best_params_}")
    best_estimators[name] = grid.best_estimator_
    res = grid.cv_results_
    bi = grid.best_index_
    cv_best_rows.append({
        "Model": name,
        "Best Params": grid.best_params_,
        "Mean Accuracy": res["mean_test_accuracy"][bi],
        "Mean Precision": res["mean_test_precision"][bi],
        "Mean Recall": res["mean_test_recall"][bi],
        "Mean F1": res["mean_test_f1"][bi],
        "Std F1": res["std_test_f1"][bi],
        "Mean Fit Time (s)": res["mean_fit_time"][bi],
    })
    for i in range(len(res["params"])):
        cv_rows.append({
            "Model": name,
            "Params": res["params"][i],
            "Mean Accuracy": res["mean_test_accuracy"][i],
            "Mean Precision": res["mean_test_precision"][i],
            "Mean Recall": res["mean_test_recall"][i],
            "Mean F1": res["mean_test_f1"][i],
            "Std F1": res["std_test_f1"][i],
            "Mean Fit Time (s)": res["mean_fit_time"][i],
        })

df_cv_all = pd.DataFrame(cv_rows)
df_cv_best = pd.DataFrame(cv_best_rows).sort_values("Mean F1", ascending=False)
print("\n=== Best config per model ===")
display(df_cv_best)
print("\n=== All param combinations ===")
display(df_cv_all)


## 8. CV confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for ax, (name, model) in zip(axes, best_estimators.items()):
    y_pred_cv = cross_val_predict(model, X_train, y_train, cv=5)
    cm = confusion_matrix(y_train, y_pred_cv)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(f"CV CM — {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.show()


## 9. Phase 2 — Held-out test evaluation

In [ ]:
print("=" * 60)
print("PHASE 2: Final Test Set Evaluation")
print("=" * 60)
test_rows = []
target_names = [f"Digit {i}" for i in range(10)]
for name, model in best_estimators.items():
    print(f"\n--- {name} ---")
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Test Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=target_names, digits=3))
    test_rows.append({"Model": name, "Test Accuracy": acc})
df_test = pd.DataFrame(test_rows).sort_values("Test Accuracy", ascending=False)
display(df_test)


## 10. Comparison charts

In [ ]:
df_plot = df_cv_best.merge(df_test, on="Model")
colors = ["#7e57c2", "#26a69a", "#42a5f5", "#ef5350"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bars = axes[0].bar(df_plot["Model"], df_plot["Mean Accuracy"], color=colors[:len(df_plot)])
axes[0].set_title("Mean CV Accuracy (5-fold)", fontweight="bold")
axes[0].set_ylim(0, 1)
axes[0].grid(axis="y", linestyle="--", alpha=0.6)
for b in bars:
    h = b.get_height()
    axes[0].annotate(f"{h:.1%}", (b.get_x()+b.get_width()/2, h), ha="center", va="bottom", fontsize=10, fontweight="bold")
bars2 = axes[1].bar(df_plot["Model"], df_plot["Test Accuracy"], color=colors[:len(df_plot)])
axes[1].set_title("Held-out Test Accuracy", fontweight="bold")
axes[1].set_ylim(0, 1)
axes[1].grid(axis="y", linestyle="--", alpha=0.6)
for b in bars2:
    h = b.get_height()
    axes[1].annotate(f"{h:.1%}", (b.get_x()+b.get_width()/2, h), ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df_plot["Mean Fit Time (s)"], df_plot["Mean Accuracy"], s=120, c=colors[:len(df_plot)])
for _, r in df_plot.iterrows():
    ax.annotate(r["Model"], (r["Mean Fit Time (s)"], r["Mean Accuracy"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set_xlabel("Mean fit time per fold (s)")
ax.set_ylabel("Mean CV Accuracy")
ax.set_title("Accuracy vs Training Cost", fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 11. Summary

| Rank | Model | Typical CV Acc |
|------|-------|----------------|
| 1 | **MLP** | ~0.80–0.85 |
| 2 | **LogisticReg** | ~0.74–0.78 |
| 3 | **LinearReg_OvA** | ~0.64–0.68 |
| 4 | **NaiveBayes** | ~0.50–0.55 |

Next steps: CNN, data augmentation, broader MLP hyperparameter search.
